# F7 – Anomalies sur le S&P 500

**Nom et prénom :** LARHMAM CHAHD    **Code Apogée :** 25007578    **Groupe :** FIN3
    **Filière :** S7 FIN / S7 CI    **Date :** 17/11/2026

**Dépôt GitHub :** https://github.com/https://github.com/chahdlarhmamencg-sys/25007578-LARHMAM-CHAHD-FIN3.git
## 1. Sujet et problématique
- **Intérêt du sujet** (2 à 3 lignes) : Un gérant de risque ou une salle de marché doit repérer vite les journées où le marché se comporte de façon inhabituelle, pour réduire l'exposition, déclencher une revue du risque ou ajuster une limite de perte. Un détecteur simple et automatique aide cette décision.
- **Problématique** (une question) : Un Isolation Forest ajusté sur le passé du S&P 500 repère-t-il, sur une période postérieure, au moins autant de journées atypiques documentées que la règle des trois écarts-types ?
- **Objectif mesurable** (métrique, référence, période de test, seuil) : Métrique principale : nombre d'événements documentés (liste de 5, fixée avant les résultats) retrouvés à ± 2 jours de bourse parmi les jours signalés. Référence : règle des 3σ calibrée sur l'apprentissage. Période de test : [du 13/11/2023 au 05/11/2026 (748 jours)]. Seuil de réussite : l'Isolation Forest retrouve au moins autant d'événements que la règle des 3σ avec au plus 5 % de jours signalés sur le test.
- **Nature du problème** (et pourquoi) : Détection d'anomalies non supervisée : aucun jour n'est étiqueté « anormal » dans les données, il n'y a donc pas de variable cible.
- **Algorithme principal envisagé** (n° de l'annexe A) : 24 – Isolation Forest

In [25]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import sklearn

GRAINE = 42
np.random.seed(GRAINE)
print(pd.__version__, np.__version__, sklearn.__version__)

2.2.3 2.4.6 1.6.1


2. Jeu de données
## 2. Jeu de données

- **Nom :** S&P 500 (série SP500), niveau de clôture journalier de l'indice.
- **Source exacte et lien :** Federal Reserve Bank of St. Louis, base FRED. URL : https://fred.stlouisfed.org/series/SP500
- **Date d'extraction :** 17/11/2026 [à corriger si tu as téléchargé le fichier un autre jour]. La dernière observation du fichier est le 05/11/2026.
- **Licence / conditions d'utilisation :** [à compléter après lecture de la page FRED : la série provient de S&P Dow Jones Indices et FRED y indique des restrictions de diffusion. Recopie la mention exacte. Si rien n'est indiqué, écris « aucune condition indiquée sur la page de la série ».]
- **Taille :** 2 609 lignes et 2 colonnes (`observation_date`, `SP500`), du 07/11/2016 au 05/11/2026, soit environ 10 ans.
- **Fréquence :** quotidienne, pour les jours ouverts de la bourse américaine. Les écarts entre dates sont de 1 jour, ou de 3 jours autour des week-ends.
- **Unité d'observation :** une ligne représente un jour de bourse, avec le niveau de clôture de l'indice.
- **Variable cible :** aucune. Le problème est non supervisé : aucun jour n'est étiqueté « anormal » dans les données. Les anomalies sont les jours isolés par l'algorithme, comparés ensuite à des événements de marché documentés.
- **Adéquation :** une série de clôtures journalières permet de calculer les rendements, la volatilité récente et le recul par rapport au sommet, c'est-à-dire des variables qui décrivent à quel point une journée s'écarte du comportement habituel du marché. Elle couvre plusieurs épisodes de stress (Covid en 2020, hausse des taux en 2022, droits de douane en 2025), ce qui permet de confronter les alertes à des événements réels.
- **Limites :**
  - environ 10 ans seulement : la crise de 2008 n'est pas couverte ;
  - 96 valeurs manquantes (3,7 %), qui correspondent à des jours fériés américains ou à des fermetures, et non à des erreurs ;
  - uniquement la clôture : pas de plus haut, de plus bas ni de volume, donc aucune volatilité intrajournalière ;
  - aucune étiquette fiable d'anomalie : la liste d'événements documentés est subjective et incomplète ;
  - la série est dominée par quelques régimes (Covid, hausse des taux, droits de douane), ce qui limite la généralisation.

**Dictionnaire des variables**

| Variable | Description | Type | Unité | Rôle | Manquants (%) |
|---|---|---|---|---|---|
| observation_date | Date de la séance de bourse | date | jour | date (index) | 0 |
| SP500 | Niveau de clôture de l'indice S&P 500 | numérique | points d'indice | brute | 3,7 (jours fériés) |
| r | Rendement logarithmique : ln(P_t / P_{t-1}) | numérique | sans unité (×100 = %) | explicative (créée) | 0 après nettoyage |
| vol20 | Écart-type des 20 derniers rendements, fenêtre se terminant en t | numérique | sans unité | explicative (créée) | 0 après nettoyage |
| r5 | Somme des 5 derniers rendements journaliers, fenêtre se terminant en t | numérique | sans unité | explicative (créée) | 0 après nettoyage |
| dd60 | Recul par rapport au plus haut des 60 derniers jours : P_t / max(P_{t-59..t}) − 1 | numérique | sans unité (≤ 0) | explicative (créée) | 0 après nettoyage |

In [26]:
px = brut.dropna(subset=["SP500"]).set_index("observation_date")["SP500"]
rend = np.log(px).diff().dropna()

print("Jours de bourse :", len(px))
print(px.describe())
print((rend * 100).describe())
print("Asymétrie :", stats.skew(rend), "| Kurtosis en excès :", stats.kurtosis(rend))
print((rend.abs() * 100).sort_values(ascending=False).head(10))

Jours de bourse : 2513
count    2513.000000
mean     4213.982591
std      1515.384160
min      2131.520000
25%      2867.190000
50%      3999.090000
75%      5209.910000
max      7818.930000
Name: SP500, dtype: float64
count    2512.000000
mean        0.049298
std         1.145552
min       -12.765214
25%        -0.376751
50%         0.080967
75%         0.588000
max         9.089490
Name: SP500, dtype: float64
Asymétrie : -0.6935591065089358 | Kurtosis en excès : 16.562206228142728
observation_date
2020-03-16    12.765214
2020-03-12     9.994485
2025-04-09     9.089490
2020-03-24     8.968316
2020-03-13     8.880836
2020-03-09     7.901039
2020-04-06     6.796823
2025-04-04     6.160907
2020-06-11     6.075276
2020-03-26     6.054383
Name: SP500, dtype: float64


In [ ]:
from google.colab import files
files.upload()

In [ ]:
px = brut.dropna(subset=["SP500"]).set_index("observation_date")["SP500"]
rend = np.log(px).diff().dropna()

print("Jours de bourse :", len(px))
print(px.describe())
print((rend * 100).describe())
print("Asymétrie :", stats.skew(rend), "| Kurtosis en excès :", stats.kurtosis(rend))
print((rend.abs() * 100).sort_values(ascending=False).head(10))

In [ ]:
brut = pd.read_csv("SP500 (2).csv", parse_dates=["observation_date"])
print(brut.shape)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(px)
plt.title("S&P 500 : niveau de clôture journalier")
plt.xlabel("Date")
plt.ylabel("Indice (points)")
plt.show()

**Interprétation du graphique 1.** Sur dix ans, l'indice passe d'environ 2 130 points (novembre 2016) à plus de 7 800 points, avec une forte tendance haussière. Cette tendance est interrompue par trois épisodes : un effondrement brutal en mars 2020 (l'indice tombe autour de 2 240 points), une correction progressive en 2022 et un nouveau repli net début avril 2025. Le niveau de l'indice évolue avec le temps et ne reste pas autour d'une valeur stable (la série est non stationnaire) : on ne peut donc pas y repérer des journées anormales directement. On travaillera sur les rendements journaliers, qui mesurent le choc de chaque jour.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(rend * 100)
plt.title("S&P 500 : rendements journaliers (log)")
plt.xlabel("Date")
plt.ylabel("Rendement (%)")
plt.show()

Interprétation du graphique 2.

 Les rendements oscillent autour de 0 avec des périodes calmes et des périodes agitées (volatilité qui se regroupe). Les plus gros chocs se concentrent en mars 2020 (jusqu’à -12,8 %) et en avril 2025. Ces valeurs extrêmes sont réelles, je ne les supprime pas : ce sont justement les journées que le détecteur doit repérer.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(rend * 100, bins=100, density=True)
x = np.linspace(-6, 6, 300)
plt.plot(x, stats.norm.pdf(x, rend.mean() * 100, rend.std() * 100))
plt.title("Distribution des rendements vs loi normale")
plt.xlabel("Rendement (%)")
plt.ylabel("Densité")
plt.show()

la distribution a des queues bien plus épaisses que la loi normale (kurtosis en excès de 16,6, asymétrie de -0,69). Une règle des 3σ, qui suppose des rendements à peu près normaux, sera donc peu adaptée. C’est ce qui motive la comparaison avec l’Isolation Forest.

In [ ]:
print("Valeurs manquantes :", brut["SP500"].isna().sum())
print("Doublons de dates :", brut["observation_date"].duplicated().sum())
print("Lignes avant :", len(brut), "| Lignes après suppression des manquants :", len(px))

Qualité des données.

Valeurs manquantes : détectées avec isna(). Elles correspondent aux jours fériés où la bourse américaine est fermée (la ligne existe mais sans valeur). Je les supprime avec dropna, car il n’y a rien à analyser ce jour-là : on passe de 2 609 lignes à 2 513 jours de bourse.
Doublons : détectés avec duplicated() sur la colonne des dates. Il n’y en a aucun, donc aucune action nécessaire.
Valeurs atypiques : les très gros rendements (par exemple -12,8 % en mars 2020) sont conservés. Ils sont réels, liés à de vrais événements de marché, et ce sont justement les journées que le détecteur doit repérer. Les supprimer fausserait le projet.

## 4. Préparation

In [ ]:
df = pd.DataFrame({"rend": rend})

# volatilité sur les 20 derniers jours, fenêtre qui se termine à la date t
df["vol20"] = df["rend"].rolling(20).std()

# recul par rapport au sommet atteint jusqu'à la date t (valeur <= 0)
df["drawdown"] = px.loc[df.index] / px.loc[df.index].cummax() - 1

# on retire les lignes incomplètes (les 19 premiers jours n'ont pas de vol20)
df = df.dropna()

print(df.shape)
df.head()

In [ ]:
coupe = int(len(df) * 0.70)
train = df.iloc[:coupe]
test = df.iloc[coupe:]

print("Apprentissage :", train.index[0].date(), "->", train.index[-1].date(), "|", len(train), "jours")
print("Test          :", test.index[0].date(), "->", test.index[-1].date(), "|", len(test), "jours")

Préparation des données.

Nettoyage : les jours fériés (valeurs manquantes) ont déjà été supprimés. Je travaille sur les rendements et non sur le niveau de l’indice, car le niveau n’est pas stationnaire.
Variables créées : (1) rend = ln(P_t / P_{t-1}), le choc du jour ; (2) vol20 = écart-type des 20 derniers rendements, fenêtre finissant à t, qui mesure l’agitation récente ; (3) drawdown = P_t / max(P_s, s ≤ t) − 1, le recul par rapport au sommet passé. Les 19 premières lignes, incomplètes, sont retirées.
Prévention des fuites : chaque variable n’utilise que des informations disponibles à la date t (aucun shift négatif, fenêtres terminées à t). Le découpage est chronologique, sans mélange : apprentissage du [date début] au [date fin], test du [date début] au [date fin]. Le modèle est ajusté sur l’apprentissage seulement et noté sur le test, une seule fois.
Mise à l’échelle : les trois variables n’ont pas la même unité (rendement, volatilité, recul), donc je les standardise avec un StandardScaler placé dans un Pipeline et ajusté uniquement sur l’apprentissage (fait en section 6). Une règle des 3σ s’appuie aussi sur la moyenne et l’écart-type de l’apprentissage seulement.

In [ ]:
df = pd.DataFrame({"rend": rend})
df["vol20"] = df["rend"].rolling(20).std()
df["drawdown"] = px.loc[df.index] / px.loc[df.index].cummax() - 1
df = df.dropna()
print(df.shape)

coupe = int(len(df) * 0.70)
train = df.iloc[:coupe]
test = df.iloc[coupe:]

print("Apprentissage :", train.index[0].date(), "->", train.index[-1].date(), "|", len(train), "jours")
print("Test          :", test.index[0].date(), "->", test.index[-1].date(), "|", len(test), "jours")